A list of differences of usual programming languages vs Python:

**Classes are objects, instances of `type`, and you can build them by calling `type(name, bases, dict)` at runtime.** No `class` statement required. Java has `Class<?>` as a reflective handle but you cannot construct a new class by calling something. The idea that the class system is itself made of regular objects is the deepest reframe.

**Class definitions are executed top-to-bottom as ordinary code.** A `class` body is a script that runs in a fresh namespace; the resulting namespace becomes the class. You can put `print`, `if`, `for` in there. Classes are _built_, not _declared_. Java parses a class; Python runs one.

**Everything is looked up by name at runtime, including method calls.** `obj.method()` is "fetch the attribute named `method`, then call it." There is no vtable, just a dict. Methods can be swapped on live objects and live classes. Monkey-patching isn't a hack, it's the natural consequence.

**Attribute access goes through an interceptable protocol.** `__getattr__`, `__getattribute__`, `__setattr__`, plus descriptors (`__get__`/`__set__`) on class attributes. What looks like a field read in Java is a function call you can hook in Python. Properties, ORMs, and lazy attributes all fall out of this.

**Read and write of an outer-scope name are asymmetric.** Reading `count` inside a function falls back through enclosing scopes to globals to builtins. Writing `count = ...` silently makes a local. Mutating a global requires `global count`. The same asymmetry shows up with `self.x` reading falling back to the class but `self.x = ...` always landing on the instance. No analog in Java or C, where declarations make the scope of every name explicit.

**Assignment inside a function makes the name local for the _entire_ function body, even before the assignment line.** Reading a global, then assigning to it later in the same function, raises `UnboundLocalError` on the _read_. The locality decision is made at compile time from the presence of any assignment anywhere.

**Methods are just functions; `self` is a parameter, not a keyword.** The "method-ness" comes from the descriptor protocol binding the function at lookup time. You can pull a method off a class and call it as a plain function. Assigning a function to an instance attribute does _not_ make it a bound method, because the descriptor protocol only fires for class attributes — its own jaw-dropper.

**Class attributes are shared by default; "instance fields" only exist once you assign to `self.x`.** In Java, fields are declared on the class but each instance gets its own slot. In Python, `class Foo: x = 0` creates one `x` on the class, visible through every instance, until an instance assigns to its own `x` and shadows it. The mutable-class-attribute footgun (`items = []` on the class, every instance shares the list) lives here.

**Functions are objects with attributes you can set.** `def f(): pass; f.calls = 0`. They have a `__dict__`. Decorators exploit this constantly. In Java a method is not a value, let alone one with fields.

**`+=` on a list inside a tuple raises `TypeError` _and_ mutates.** `t = ([1, 2],); t[0] += [3]` throws but `t` is now `([1, 2, 3],)`. An operation that fails and succeeds simultaneously has no analog in either language.

**Mutable default arguments.** `def f(items=[])` evaluates `[]` once at def-time; every call shares the same list. Looks like memory corruption to a C programmer.

**Operators are dunder methods.** `a + b` is `a.__add__(b)`. `==`, `<`, `[]`, `()`, `in`, `with`, `len()` — all dispatch to methods. C++ has operator overloading as a special syntactic feature; in Python it's uniformly "operator is sugar for method call."

**`import` is a statement that executes a file, once, and caches the module object.** The module is an object with attributes; you can mutate it after importing. `from X import y` copies the _current_ binding, so later reassignments in `X` don't propagate. Surprises anyone who thinks `import` resembles `#include` or Java's `import`.

**Variables don't have types; values do.** A name is a binding in a namespace dict, not a typed slot. `x = 1; x = "hello"; x = SomeClass` is fine. `del x` removes the binding entirely.

**Namespaces are (mostly) dicts you can inspect and modify.** `globals()`, `locals()`, `vars(obj)`, `obj.__dict__`, `MyClass.__dict__`. A module's global scope is literally a dict on the module object.

**There are no private attributes.** Leading-underscore is convention. Double-underscore name-mangles (`__x` → `_ClassName__x`) to avoid inheritance collisions, not to enforce privacy. Encapsulation is a social contract.

**Class body names are invisible to methods and comprehensions inside the class.** Methods must write `self.x` or `ClassName.x`; comprehensions in a class body can't see sibling class attributes at all. Contradicts the Java model of "everything in the class sees everything in the class."

**Iteration, context managers, and most language constructs are protocols.** `for` calls `iter()` then `next()` until `StopIteration`. `with` calls `__enter__`/`__exit__`. Anything implementing the protocol participates. Language constructs are thin wrappers over method calls.

**Decorators are just function application.** `@decorator` above a `def` means `f = decorator(f)`. The `@` is sugar; there's no special "decorator" concept.

**`bool` is a subclass of `int`.** `True + True == 2`. `sum([True, False, True]) == 2`. Java's `boolean` is a distinct primitive. C's `_Bool` is closer but still doesn't make `True + True` feel right.

**Iterators are silently exhausted after one pass.** Build a `zip` or generator, iterate it for a check, then iterate it again for the real work — the second pass is empty, no error.

**`is` vs `==`, with small-int caching making `is` accidentally "work" for small numbers.** `a = 256; b = 256; a is b` → `True`; `257` → `False`. Java has the exact analog with `Integer` autoboxing's -128..127 cache, so a Java programmer has seen this movie.

**Tuple-with-one-element needs a trailing comma; the comma makes the tuple, not the parens.** `(1)` is an int, `(1,)` is a tuple, `1, 2, 3` with no parens is also a tuple.

**`copy` vs `deepcopy`, and assignment never copies.** `b = a` shares the object. Java is identical for objects so a Java programmer is fine; C programmers used to `struct` assignment will be briefly confused.

**Truthiness of empty collections.** `if not x:` fires for `None`, `0`, `""`, `[]`, `{}`. Java requires an actual `boolean`. Mostly a "be careful" issue.

**Chained comparisons.** `1 < x < 10` means `1 < x and x < 10`. Java/C parse it as `(1 < x) < 10` and give nonsense. Python's behavior is actually the _less_ surprising one once seen.

**`__init__` is not the constructor; `__new__` is.** Matters mainly when subclassing immutable types. Java's constructors do both steps in one, so the split feels academic until you hit it.

**`global` and `nonlocal` keywords exist.** No Java analog (no nested functions, no module globals in the same sense), but the keywords are explicit and well-named — unfamiliar rather than surprising.

**`0.1 + 0.2 != 0.3`.** IEEE 754, identical in Java and C. Not a Python oddity at all, just a floating-point one.